In [1]:
import os
import numpy as np
import flopy
import math
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [2]:
####################
### set up paths ###
####################

notebook_dir = os.getcwd()
mf6_exe = os.path.join(notebook_dir, "..", "binaries", "MODFLOW6", "windows", "mf6.exe")

workspace = "mf6-com-1"
if not os.path.exists(workspace):
    os.makedirs(workspace)

################################
### random domain generation ###
################################

cell_dim = float(np.random.uniform(1.0, 5.0)) # square cells between 1.0 and 5.0 units
target_lx = np.random.uniform(50, 150) # target domain size between 50-150 units in x
target_ly = np.random.uniform(50, 150) # target domain size between 50-150 units in y

ncol, nrow = int(target_lx / cell_dim), int(target_ly / cell_dim) # number of columns and rows
lx, ly = float(ncol * cell_dim), float(nrow * cell_dim) # adjust domain size to be an integer number of cells

dt = 7.0 # stress period length in days
nper = 52 # number of stress periods (e.g. 52 weeks in a year)
time_steps = np.arange(dt, (nper * dt) + dt, dt) # time steps for plotting, from dt to nper*dt with step of dt

########################################
### define simulation setup function ###
########################################

def setup_sim(name, wel_data):
    sim = flopy.mf6.MFSimulation(sim_name=name, exe_name=mf6_exe, sim_ws=workspace)
    flopy.mf6.ModflowTdis(sim, nper=nper, perioddata=[(dt, 1, 1.0)]*nper)
    gwf = flopy.mf6.ModflowGwf(sim, modelname=name, save_flows=True)
    
    # centre the grid
    flopy.mf6.ModflowGwfdis(gwf, nlay=1, nrow=nrow, ncol=ncol, 
                            delr=cell_dim, delc=cell_dim, 
                            xorigin=-lx/2, yorigin=-ly/2)
    
    flopy.mf6.ModflowIms(sim, complexity="SIMPLE") # use the SIMPLE solver for simplicity
    flopy.mf6.ModflowGwfnpf(gwf, k=1e-5, icelltype=0) # confined aquifer with hydraulic conductivity of 1e-5 units
    flopy.mf6.ModflowGwfsto(gwf, ss=1e-7, transient=True) # specific storage of 1e-7 per unit head change
    flopy.mf6.ModflowGwfic(gwf, strt=0.0) # initial head of 0 everywhere
    flopy.mf6.ModflowGwfwel(gwf, stress_period_data=wel_data)
    flopy.mf6.ModflowGwfoc(gwf, head_filerecord=f"{name}.hds", saverecord=[("HEAD", "ALL")])
    return sim, gwf

#################################
### compliance ring and wells ###
#################################

# geometry and coordinate snapping logic using a dummy model to access the grid object
_, dummy_gwf = setup_sim("dummy", {})
grid = dummy_gwf.modelgrid
xc, yc = grid.xcellcenters, grid.ycellcenters

# define compliance and no-go radii based on the snapped domain size
compliance_radius = float(min(lx, ly) / 2 * 0.95) # compliance ring is 95% of the max inscribed circle radius
no_go_radius = compliance_radius * 0.2 # no-go zone is 20% of the compliance radius
thetas = np.linspace(0, 2 * np.pi, 12, endpoint=False)

# snap compliance points to cell centres
cp_cells = []
for t in thetas:
    r, c = grid.intersect(compliance_radius * np.cos(t), compliance_radius * np.sin(t))
    cp_cells.append((r, c))

centre_r, centre_c = grid.intersect(0.0, 0.0)

# snap wells to cell centres, ensuring they are within the compliance ring and outside the no-go zone
n_wells = 3
wells = []
while len(wells) < n_wells:
    wx, wy = np.random.uniform(-lx/2, lx/2), np.random.uniform(-ly/2, ly/2)
    dist = np.sqrt(wx**2 + wy**2)
    if no_go_radius < dist < compliance_radius:
        r, c = grid.intersect(wx, wy)
        if (r, c) not in [(w['r'], w['c']) for w in wells]:
            wells.append({'r': r, 'c': c, 'Q': np.random.uniform(15, 55, nper)})

################
### analysis ###
################

# run the random well scenario
# generates the average drawdown at the compliance points per stress period for the random pumping scenario at the wells

wel_spd_r = {p: [((0, w['r'], w['c']), -w['Q'][p]) for w in wells] for p in range(nper)}
sim_r, _ = setup_sim("random_target", wel_spd_r)
sim_r.write_simulation()
sim_r.run_simulation(silent=True)

hds_r = flopy.utils.binaryfile.HeadFile(os.path.join(workspace, "random_target.hds")).get_alldata()
s_target = [np.mean([0.0 - hds_r[p, 0, r, c] for r, c in cp_cells]) for p in range(nper)]
s_incremental_target = np.diff(s_target, prepend=0)

# run the unit response (Q = 1 at centre well) scenario 
# generates the drawdown at the compliance points per unit pumping at the centre well

wel_spd_u = {p: [((0, centre_r, centre_c), -1.0)] for p in range(nper)}
sim_u, _ = setup_sim("unit_response", wel_spd_u)
sim_u.write_simulation()
sim_u.run_simulation(silent=True)

hds_u = flopy.utils.binaryfile.HeadFile(os.path.join(workspace, "unit_response.hds")).get_alldata()
U = np.array([np.mean([0.0 - hds_u[p, 0, r, c] for r, c in cp_cells]) for p in range(nper)])

# conduct deconvolution
# finds the equivalent pumping rate time series at the centre well 
# to match the target drawdown at the compliance points
q_equiv = np.zeros(nper)
delta_q = np.zeros(nper)

for k in range(nper):
    # prev_impact is the drawdown caused at time k by all previous changes (delta_q)
    prev_impact = sum(delta_q[j] * U[k-j] for j in range(k))
    # required change in Q to meet the target drawdown at time k, accounting for previous impacts
    # divide by U[0] since it's the immediate response to a change in Q 
    delta_q[k] = (s_incremental_target[k] - prev_impact) / U[0] 
    q_equiv[k] = np.sum(delta_q[:k+1])

writing simulation...
  writing simulation name file...
  writing simulation tdis package...
  writing solution package ims_-1...
  writing model random_target...
    writing model name file...
    writing package dis...
    writing package npf...
    writing package sto...
    writing package ic...
    writing package wel_0...
INFORMATION: maxbound in ('gwf6', 'wel', 'dimensions') changed to 3 based on size of stress_period_data
    writing package oc...
writing simulation...
  writing simulation name file...
  writing simulation tdis package...
  writing solution package ims_-1...
  writing model unit_response...
    writing model name file...
    writing package dis...
    writing package npf...
    writing package sto...
    writing package ic...
    writing package wel_0...
INFORMATION: maxbound in ('gwf6', 'wel', 'dimensions') changed to 1 based on size of stress_period_data
    writing package oc...


In [3]:
#####################
### visualisation ###
#####################

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.1,
                    subplot_titles=("Weekly Drawdown at Compliance Ring", "Weekly Pumping Rates"))

# row 1: drawdown at compliance points - target vs equivalent centre well match
fig.add_trace(go.Scatter(x=time_steps, y=s_incremental_target, name=f"Superposition of {n_wells} Random Wells Drawdown (Target)", 
                         line=dict(color='green', width=4)), row=1, col=1)
fig.add_trace(go.Scatter(x=time_steps, y=s_incremental_target, name="Equivalent Centre Well (Match)", 
                         line=dict(color='red', dash='dot', width=2)), row=1, col=1)

# row 2: pumping rates of the random wells and the equivalent centre well
for i, w in enumerate(wells):
    fig.add_trace(go.Scatter(x=time_steps, y=w['Q'], name=f"Well {i+1}", 
                             line=dict(width=1), opacity=0.4), row=2, col=1)

fig.add_trace(go.Scatter(x=time_steps, y=q_equiv, name="Centre Well Pumping Rate", 
                         line=dict(color='black', width=3)), row=2, col=1)

fig.update_layout(height=800, template="plotly_white", 
                  title=(f"Multi-Well Numerical Analysis: {n_wells} Individual Wells vs Equivalent Centre Well Responses"))
fig.update_yaxes(title_text="Drawdown (m)", row=1, col=1)
fig.update_yaxes(title_text="Pumping Rate (m³/d)", row=2, col=1)

fig.show()

In [4]:
# generate grid lines for plotting
x_edges = np.linspace(-lx/2, lx/2, ncol + 1)
y_edges = np.linspace(-ly/2, ly/2, nrow + 1)

grid_x, grid_y = [], []
for x in x_edges:
    grid_x.extend([float(x), float(x), None])
    grid_y.extend([float(y_edges[0]), float(y_edges[-1]), None])
for y in y_edges:
    grid_x.extend([float(x_edges[0]), float(x_edges[-1]), None])
    grid_y.extend([float(y), float(y), None])

fig = go.Figure()

# plot domain boundary
fig.add_shape(type="rect", x0=-lx/2, y0=-ly/2, x1=lx/2, y1=ly/2, 
              line=dict(color="black"), opacity=0.1)

# plot grid lines
fig.add_trace(go.Scatter(x=grid_x, y=grid_y, mode='lines', 
                         line=dict(color='silver', width=0.5), 
                         name=f"MODFLOW Grid ({nrow}x{ncol})", hoverinfo='skip'))

# plot compliance ring
t_fine = np.linspace(0, 2 * np.pi, 100)
fig.add_trace(go.Scatter(x=compliance_radius*np.cos(t_fine), 
                         y=compliance_radius*np.sin(t_fine),
                         mode='lines', line=dict(color='blue', dash='dot', width=1), 
                         name="Compliance Ring"))

# plot compliance points
cp_x = [xc[r, c] for r, c in cp_cells]
cp_y = [yc[r, c] for r, c in cp_cells]
fig.add_trace(go.Scatter(
    x=cp_x, y=cp_y, mode='markers', 
    marker=dict(color='yellow', size=10, symbol='star', line=dict(width=1, color='black')), 
    name="Compliance Point"
))

# plot the centre well
fig.add_trace(go.Scatter(
    x=[xc[centre_r, centre_c]], y=[yc[centre_r, centre_c]], mode='markers+text', 
    text=["Centre Well"], textposition="top center",
    marker=dict(color='black', size=12, symbol='hexagon'), 
    name="Centre Well"
))

# plot the random wells
colours = ['#636EFA', '#EF553B', '#00CC96', '#AB63FA', '#FFA15A', '#19D3F3']
for i, w in enumerate(wells):
    color = colours[i % len(colours)]
    fig.add_trace(go.Scatter(
        x=[xc[w['r'], w['c']]], y=[yc[w['r'], w['c']]], mode='markers+text', 
        text=[f"W{i+1}"], textposition="top center",
        marker=dict(color=color, size=10, line=dict(width=1, color='white')), 
        name=f"Well {i+1}"
    ))

# pretty up the layout
plot_limit = max(lx, ly) / 2 + 5
fig.update_layout(
    title=f"Multi-Well Numerical Analysis: {n_wells} Individual Wells and a Centre Well in a {nrow}x{ncol} MODFLOW Grid",
    xaxis=dict(title="X Coordinate (m)", range=[-plot_limit, plot_limit], tickformat=".1f"),
    yaxis=dict(title="Y Coordinate (m)", range=[-plot_limit, plot_limit], tickformat=".1f",
               scaleanchor="x", scaleratio=1),
    template="plotly_white", width=800, height=800,
    legend=dict(yanchor="top", y=0.99, xanchor="left", x=1.02)
)

fig.show()